# SPS-CA Colab

Run the current `main` repository, keep Ollama/Web UI, and test the existing 1000-scenario suite without generating new scenarios.

In [ ]:
# CELL 1 — ENVIRONMENT + OLLAMA
import os, subprocess, sys, time
MODEL = 'qwen3-coder:30b'
def run(cmd, check=True):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout)
    if check and r.returncode: raise RuntimeError(r.stdout)
run('apt-get update -qq')
run('apt-get install -y -qq git curl zstd')
run(f'{sys.executable} -m pip install -q --upgrade pip')
run(f'{sys.executable} -m pip install -q pyngrok pytest pytest-json-report')
if subprocess.run('command -v ollama', shell=True, capture_output=True).returncode != 0: run('curl -fsSL https://ollama.com/install.sh | sh')
run('ollama --version')
subprocess.run("pkill -f 'ollama serve' 2>/dev/null || true", shell=True)
subprocess.Popen(['ollama','serve'], stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
for _ in range(30):
    time.sleep(1)
    if subprocess.run('curl -s http://127.0.0.1:11434/api/tags', shell=True, capture_output=True).returncode == 0: break
else: raise RuntimeError('Ollama service did not become ready')
run(f'ollama pull {MODEL}')
run('ollama list')
print('CELL 1 PASSED')

In [ ]:
# CELL 2 — PULL LATEST MAIN + INSTALL REQUIREMENTS
import os, shutil, subprocess, sys
from pathlib import Path
REPO = Path('/content/SPS_CA')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch','main','--single-branch','https://github.com/muhammadnaumantahir/SPS_CA.git',str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements.txt')], check=True)
print('Commit:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())
print('✓ Canonical main pulled')

In [ ]:
# CELL 3 — CORE SMOKE CHECK
import os, sys
os.chdir('/content/SPS_CA')
sys.path.insert(0,'/content/SPS_CA') if '/content/SPS_CA' not in sys.path else None
from layers.architecture import architecture_manifest
from core.canonical_sps_pipeline import CanonicalSPSPipeline
from layers.layer_08_evolution_core import GrowthDecisionEngine
m = architecture_manifest()
assert len(m['layers']) == 10 and m['brain']['name'] == 'SPS-CA Brain'
print('✓ Ten layers, Brain, pipeline, and Layer 8 available')

In [ ]:
# CELL 4 — SPS-CA WEB UI + NGROK
import os, subprocess, sys, time
from getpass import getpass
from pyngrok import ngrok
os.chdir('/content/SPS_CA'); PORT = 5000
subprocess.run("pkill -f 'ui.web_app' 2>/dev/null || true", shell=True)
log = open('/tmp/sps_ca_web_ui.log','w')
server = subprocess.Popen([sys.executable,'-m','ui.web_app'], cwd='/content/SPS_CA', stdout=log, stderr=subprocess.STDOUT)
time.sleep(2)
token = os.environ.get('NGROK_AUTHTOKEN','')
if not token:
    try: token = getpass('Enter NGROK_AUTHTOKEN (blank = local UI): ').strip()
    except Exception: token = ''
if token:
    ngrok.set_auth_token(token); tunnel = ngrok.connect(PORT,'http'); print('SPS-CA public Web UI:', tunnel.public_url)
else: print('SPS-CA local Web UI: http://127.0.0.1:5000')
print('✓ Web UI started'); print('✓ Ollama: http://127.0.0.1:11434')

In [ ]:
# CELL 5 — TEST EXISTING 1000 SCENARIOS (NO GENERATION)
import json, subprocess, sys
from pathlib import Path
REPO = Path('/content/SPS_CA'); SCENARIO_FILE = REPO/'evaluation/scenarios/growth.json'; REPORT_FILE = Path('/content/sps_ca_1000_test_report.json')
assert SCENARIO_FILE.exists(), SCENARIO_FILE
from scripts.evaluate_growth_scenarios import load, validate_structure, validate_routing, validate_evolution_contracts
scenarios = load(); validate_structure(scenarios); validate_routing(scenarios); validate_evolution_contracts(scenarios)
r = subprocess.run([sys.executable,'-m','pytest','layers/layer_08_evolution_core/tests/test_growth_decision_scoring.py','layers/layer_08_evolution_core/tests/test_capability_improvement.py','-q'], cwd=REPO, text=True, capture_output=True)
print(r.stdout)
if r.returncode: print(r.stderr); raise RuntimeError('Targeted evolution tests failed')
breakdown = {}
for s in scenarios: breakdown[s['scenario_type']] = breakdown.get(s['scenario_type'],0)+1
report = {'status':'PASS','total':len(scenarios),'breakdown':breakdown,'generation_called':False,'targeted_evolution_tests':'PASS','scenario_file':str(SCENARIO_FILE)}
REPORT_FILE.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
print(f"✓ Existing suite tested: {len(scenarios)}/1000")
print(f"✓ Routing: {breakdown.get('capability_routing',0)}/490")
print(f"✓ Autonomous evolution: {breakdown.get('autonomous_evolution',0)}/500")
print(f"✓ Evolution proof: {breakdown.get('evolution_proof',0)}/10")
print('✓ No scenario generation called')

In [ ]:
# CELL 6 — REPORT RESULTS
import json
from pathlib import Path
r = json.loads(Path('/content/sps_ca_1000_test_report.json').read_text(encoding='utf-8')); b = r['breakdown']
print('='*70); print('SPS-CA 1000-SCENARIO REPORT'); print('='*70)
print('Overall:', r['status']); print('Total:', f"{r['total']}/1000")
print('Capability routing:', f"{b.get('capability_routing',0)}/490")
print('Autonomous evolution:', f"{b.get('autonomous_evolution',0)}/500")
print('Evolution proof:', f"{b.get('evolution_proof',0)}/10")
print('Scenario generation called:', r['generation_called'])
print('Targeted scoring/improvement tests:', r['targeted_evolution_tests'])
print('='*70)